# Image-Based Disease Detection: Pneumonia from Chest X-Rays

**Goal:** Detect pneumonia from pediatric chest X-ray images using deep learning, as a proof-of-concept for AI-assisted radiology screening.

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) — 5,863 X-ray images (JPEG), 2 classes (NORMAL / PNEUMONIA), pre-split into train/val/test.

**Approach:** Transfer learning with a DenseNet121 backbone (pretrained on ImageNet) — the same family of architecture used in a lot of published pneumonia-detection literature (CheXNet), fine-tuned on this dataset.

**Pipeline:**
1. Setup & download data
2. Explore dataset (class balance, sample images)
3. Build a `tf.keras` data pipeline with augmentation
4. Transfer learning model (DenseNet121)
5. Train (feature extraction, then fine-tuning)
6. Evaluate (accuracy, precision, recall, AUC, confusion matrix)
7. Grad-CAM — visualize what the model is "looking at"
8. Conclusions

---


## 1. Setup & Download Data

This notebook is built for **Google Colab** (Runtime → Change runtime type → GPU).

The dataset is downloaded via `kagglehub`, which handles Kaggle authentication for you inside Colab — it will prompt you to log in with your Kaggle account the first time.


In [ ]:
# Run once per Colab session
!pip install kagglehub -q


In [ ]:
import kagglehub

# Downloads the dataset and returns the local path (cached after first run)
dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset downloaded to:", dataset_path)


In [ ]:
import os

# The Kaggle version of this dataset nests everything under chest_xray/chest_xray
for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, '').count(os.sep)
    if depth < 3:
        print(root)


In [ ]:
import glob

# Locate the actual chest_xray folder that contains train/val/test
candidates = glob.glob(os.path.join(dataset_path, '**', 'train'), recursive=True)
DATA_DIR = os.path.dirname(candidates[0])
print("Using data directory:", DATA_DIR)

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_score, recall_score, f1_score, accuracy_score)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Explore the Dataset

In [ ]:
for split, split_dir in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    normal = len(os.listdir(os.path.join(split_dir, 'NORMAL')))
    pneumonia = len(os.listdir(os.path.join(split_dir, 'PNEUMONIA')))
    print(f"{split:6s} -> NORMAL: {normal:5d}  PNEUMONIA: {pneumonia:5d}")


**Observation:** The training set is noticeably imbalanced toward PNEUMONIA (~74% of training images). We'll account for this with **class weights** during training rather than discarding data.

In [ ]:
counts = {
    'NORMAL': len(os.listdir(os.path.join(TRAIN_DIR, 'NORMAL'))),
    'PNEUMONIA': len(os.listdir(os.path.join(TRAIN_DIR, 'PNEUMONIA')))
}
fig, ax = plt.subplots()
ax.bar(counts.keys(), counts.values(), color=['#4C72B0', '#DD8452'])
ax.set_title("Training Set Class Distribution")
ax.set_ylabel("Number of Images")
plt.show()


In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    sample_files = os.listdir(cls_dir)[:4]
    for col, fname in enumerate(sample_files):
        img = plt.imread(os.path.join(cls_dir, fname))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(cls)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()


## 3. Data Pipeline

- Resize all images to 224x224 (DenseNet121's expected input size)
- Rescale pixel values to [0, 1]
- Apply light augmentation (rotation, zoom, horizontal flip) **only on the training set**, to reduce overfitting without distorting the medical features


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True, seed=42
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

print("Class indices:", train_gen.class_indices)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
y_train = train_gen.classes
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)


## 4. Model: Transfer Learning with DenseNet121

We use DenseNet121 pretrained on ImageNet as a frozen feature extractor, with a small classification head on top. This is the same backbone family used in CheXNet, one of the most cited pneumonia-detection papers.

**Two-phase training:**
1. Train only the new head, with the DenseNet121 base frozen (fast, prevents destroying pretrained features)
2. Unfreeze the top layers of DenseNet121 and fine-tune with a low learning rate (squeezes out extra performance)


In [ ]:
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

model.summary()


### Phase 1: Train the classification head (base frozen)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)


### Phase 2: Fine-tune the top layers of DenseNet121

In [ ]:
base_model.trainable = True

# Freeze all but the last ~30 layers so early, generic features stay intact
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=6,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
def combine_history(h1, h2, key):
    return h1.history[key] + h2.history[key]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['accuracy', 'auc']):
    ax.plot(combine_history(history1, history2, metric), label='train')
    ax.plot(combine_history(history1, history2, f'val_{metric}'), label='val')
    ax.axvline(x=len(history1.history[metric]) - 0.5, color='gray', linestyle='--', label='fine-tuning starts')
    ax.set_title(f'{metric.capitalize()} over epochs')
    ax.set_xlabel('Epoch')
    ax.legend()
plt.tight_layout()
plt.show()


## 5. Evaluation on the Test Set

In [ ]:
test_probs = model.predict(test_gen).flatten()
test_preds = (test_probs > 0.5).astype(int)
test_true = test_gen.classes

acc = accuracy_score(test_true, test_preds)
prec = precision_score(test_true, test_preds)
rec = recall_score(test_true, test_preds)
f1 = f1_score(test_true, test_preds)
auc = roc_auc_score(test_true, test_probs)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {auc:.3f}")
print()
print(classification_report(test_true, test_preds, target_names=['NORMAL', 'PNEUMONIA']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(test_true, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(test_true, test_probs)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.show()


**Why recall matters most here:** In a medical screening context, a **false negative** (telling a sick patient they're healthy) is far more dangerous than a false positive (an extra check for a healthy patient). Watch recall on the PNEUMONIA class closely — it's the most clinically important number in this report.

## 6. Grad-CAM: Visualizing What the Model Looks At

Grad-CAM highlights the regions of the X-ray that most influenced the model's prediction. This is a simple but powerful way to sanity-check that the model is actually looking at the lungs, not some artifact in the image corner.


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="conv5_block16_concat"):
    grad_model = tf.keras.models.Model(
        [model.get_layer(index=0).input],
        [model.get_layer(index=0).get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


# Grab one test batch and visualize Grad-CAM for a few images
test_gen.reset()
sample_batch, sample_labels = next(test_gen)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i in range(4):
    img_array = np.expand_dims(sample_batch[i], axis=0)
    heatmap = make_gradcam_heatmap(img_array, model)
    heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], IMG_SIZE).numpy().squeeze()

    axes[0, i].imshow(sample_batch[i])
    axes[0, i].set_title(f"True: {'PNEUMONIA' if sample_labels[i]==1 else 'NORMAL'}")
    axes[0, i].axis('off')

    axes[1, i].imshow(sample_batch[i])
    axes[1, i].imshow(heatmap_resized, cmap='jet', alpha=0.5)
    axes[1, i].set_title("Grad-CAM")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()


*Note: the exact `last_conv_layer_name` may need adjusting depending on the TensorFlow/Keras version — print `model.get_layer(index=0).summary()` to find the last convolutional layer name if this cell errors.*

## 7. Save the Trained Model

In [ ]:
model.save('pneumonia_densenet121.keras')
print("Model saved.")


## 8. Conclusions

- A DenseNet121 transfer-learning model, fine-tuned in two phases, classifies pediatric chest X-rays as NORMAL / PNEUMONIA with strong recall on the pneumonia class after accounting for class imbalance with class weights.
- Grad-CAM visualizations confirm the model is generally focusing on lung regions rather than image artifacts — an important sanity check before trusting any medical imaging model.
- This is a **proof-of-concept**, not a diagnostic tool — real clinical deployment would require a far larger, more diverse dataset, validation against radiologist consensus, and regulatory approval.

### Possible extensions
- Try other backbones (ResNet50, EfficientNetB0) and compare
- K-fold cross-validation for a more robust performance estimate
- Multi-class extension: bacterial vs. viral vs. normal (the filenames in this dataset encode that label)
- Deploy behind a simple Streamlit/Gradio app for interactive demo use
